# EDA Overview\n\n전체 계량기 개요 EDA입니다. 메타데이터, anomaly summary/detail CSV를 기준으로 계량기 분포, 경보 분포, 데이터 커버리지를 확인합니다.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path('/home/playdata2/final_pj/energy-platform')
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'eda'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metadata_path = PROJECT_ROOT / 'config' / 'meter_metadata.json'
summary_path = PROJECT_ROOT / 'outputs' / 'anomaly_results_summary.csv'
detail_path = PROJECT_ROOT / 'outputs' / 'anomaly_results_detail.csv'

metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
summary_df = pd.read_csv(summary_path)
detail_df = pd.read_csv(detail_path, parse_dates=['ts'])
metadata_df = pd.DataFrame([{'meter_urn': k, **v} for k, v in metadata.items()])
metadata_df.head()

## 1. 기초 통계

In [ ]:
detail_agg = detail_df.groupby('meter_urn').agg(
    start_ts=('ts', 'min'),
    end_ts=('ts', 'max'),
    row_count=('ts', 'size'),
    warning_count=('ensemble_level', lambda s: (s == 'WARNING').sum()),
    danger_count=('ensemble_level', lambda s: (s == 'DANGER').sum()),
).reset_index()

overview_df = metadata_df.merge(summary_df, on='meter_urn', how='left').merge(detail_agg, on='meter_urn', how='left')
overview_df[['meter_urn','meter_type','group_name','anomaly_target','row_count','warning','danger','error']].head(10)

## 2. 그룹 집계

In [ ]:
display(metadata_df['meter_type'].value_counts())
display(metadata_df['group_name'].value_counts().head(15))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
metadata_df['meter_type'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','salmon','seagreen'])
axes[0].set_title('Meter Type Count')
metadata_df['group_name'].value_counts().head(10).plot(kind='bar', ax=axes[1], color='slateblue')
axes[1].set_title('Top 10 Group Count')
plt.tight_layout()
plot_path = OUTPUT_DIR / 'overview_group_counts.png'
plt.savefig(plot_path, dpi=150)
plt.close(fig)
from IPython.display import Image, display
display(Image(filename=str(plot_path)))


## 3. 이상탐지 요약

In [ ]:
top_danger = summary_df.sort_values(['danger', 'warning'], ascending=False)[['meter_urn','danger','warning','danger_pct','warning_pct','error']].head(10)
top_warning = summary_df.sort_values(['warning', 'danger'], ascending=False)[['meter_urn','danger','warning','danger_pct','warning_pct','error']].head(10)
display(top_danger)
display(top_warning)

overview_df.to_csv(OUTPUT_DIR / 'overview_meter_summary.csv', index=False, encoding='utf-8-sig')
print('saved:', OUTPUT_DIR / 'overview_meter_summary.csv')
print('saved:', OUTPUT_DIR / 'overview_group_counts.png')